# KALIA prep v2b — permissive-licensed code rebuild + remix

Rebuilds the code shard with the permissive-license filter (MIT/Apache-2.0/BSD/ISC/Unlicense/CC0) and re-mixes with the existing TinyStories / FineWeb-Edu / Cosmopedia shards from the completed `kalia-prep-v2` run. Produces the compliance-clean `train.bin` (2.4B tokens) and `val.bin` (10M tokens) for the public model release.

In [ ]:
!pip install -q tiktoken datasets

In [ ]:
import glob
import os
import shutil

work = "/kaggle/working/kalia"
if not os.path.exists(work):
    hits = sorted(glob.glob("/kaggle/input/datasets/**/prepare.py", recursive=True))
    if not hits:
        hits = sorted(glob.glob("/kaggle/input/**/prepare.py", recursive=True))
    assert hits, "attach the kalia-code-dev dataset"
    shutil.copytree(os.path.dirname(hits[0]), work)
os.chdir(work)
print("code from", work)

In [ ]:
!mkdir -p /kaggle/working/data
!python prepare.py --source stack_smol --out /kaggle/working/data/stack_smol.bin --max-tokens 125000000 --meta /kaggle/working/data/meta_code.json
!cat /kaggle/working/data/meta_code.json

In [ ]:
import glob
import subprocess

def find(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    assert hits, f"{name} not found - attach the kalia-prep-v2 output"
    return hits[0]

shards = ",".join([
    f"{find('tinystories.bin')}:20",
    f"{find('smollm_fineweb_edu.bin')}:60",
    f"{find('cosmopedia.bin')}:15",
    "/kaggle/working/data/stack_smol.bin:5",
])
cmd = [
    "python", find("mix_bins.py"),
    "--shards", shards,
    "--val-tokens", "10000000",
    "--train-tokens", "2400000000",
    "--val-out", "/kaggle/working/data/val.bin",
    "--train-out", "/kaggle/working/data/train.bin",
    "--meta", "/kaggle/working/data/mix_meta.json",
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
!ls -lh /kaggle/working/data/
!cat /kaggle/working/data/mix_meta.json

Compliance-clean dataset ready. The v0.2.0 training kernel attaches this run's output as its kernel source.